After consulting with ChatGPT and Gemini about what to do with this dataset since having post-game information/stats is not really "predicting" a game, they explained that what is meant to do with a historical table like this in ML is to create a dataset of rolling features that consider the only previous N results in trying to predict the current game which we consider unknown. AKA, we should compute an average historical stat to use in predicting the outcome of the current game.

In [13]:
import pandas as pd
import numpy as np
df = pd.read_csv("../../data/cs2_tier1_games.csv", encoding="latin1")
df = df.sort_values(by="datetime")
df

,Unnamed: 0,match_id,game_id,tournament,team1_id,team1,team2_id,team2,score1_match,score2_match,...,team2_player4_assists,team2_player4_adr,team2_player4_kast,team2_player4_kddiff,team2_player5_kills,team2_player5_deaths,team2_player5_assists,team2_player5_adr,team2_player5_kast,team2_player5_kddiff
9071,9071,589049,-115902,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,14.0,60.7,73.2,-9.0,39.0,50.0,12.0,63.9,63.3,-11.0
9069,9069,589049,115903,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,5.0,54.0,69.6,-6.0,17.0,17.0,2.0,68.6,69.6,0.0
9068,9068,589049,115902,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,7.0,80.1,83.3,1.0,13.0,16.0,4.0,49.0,58.3,-3.0
9070,9070,589049,115901,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,2.0,48.0,66.7,-4.0,9.0,17.0,6.0,74.0,61.9,-8.0
9067,9067,589047,-115906,Thunderpick World Championship 2023,233631,Monte,233632,Wildcard Gaming,2,0,...,6.0,56.3,45.0,-15.0,14.0,29.0,2.0,53.7,47.6,-15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,140,6796580,125697,ESL Pro League Season 23,288172,FURIA Esports,288173,paiN Gaming,2,0,...,3.0,69.8,57.9,-5.0,6.0,15.0,4.0,41.9,42.1,-9.0
141,141,6796580,-125699,ESL Pro League Season 23,288172,FURIA Esports,288173,paiN Gaming,2,0,...,9.0,67.9,55.2,-15.0,15.0,29.0,8.0,47.0,49.6,-14.0
138,138,6796582,-125701,ESL Pro League Season 23,288166,G2 Esports,288167,Aurora Gaming,0,2,...,11.0,77.5,78.6,-3.0,30.0,26.0,6.0,72.4,68.2,4.0
136,136,6796582,125701,ESL Pro League Season 23,288166,G2 Esports,288167,Aurora Gaming,0,2,...,3.0,78.0,73.9,1.0,12.0,10.0,4.0,67.6,69.6,2.0


In [14]:
#Drop those that have NaN
df = df.dropna()
df

,Unnamed: 0,match_id,game_id,tournament,team1_id,team1,team2_id,team2,score1_match,score2_match,...,team2_player4_assists,team2_player4_adr,team2_player4_kast,team2_player4_kddiff,team2_player5_kills,team2_player5_deaths,team2_player5_assists,team2_player5_adr,team2_player5_kast,team2_player5_kddiff
9069,9069,589049,115903,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,5.0,54.0,69.6,-6.0,17.0,17.0,2.0,68.6,69.6,0.0
9068,9068,589049,115902,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,7.0,80.1,83.3,1.0,13.0,16.0,4.0,49.0,58.3,-3.0
9070,9070,589049,115901,Thunderpick World Championship 2023,233621,Cloud9,233622,Fnatic,2,1,...,2.0,48.0,66.7,-4.0,9.0,17.0,6.0,74.0,61.9,-8.0
9067,9067,589047,-115906,Thunderpick World Championship 2023,233631,Monte,233632,Wildcard Gaming,2,0,...,6.0,56.3,45.0,-15.0,14.0,29.0,2.0,53.7,47.6,-15.0
9066,9066,589047,115904,Thunderpick World Championship 2023,233631,Monte,233632,Wildcard Gaming,2,0,...,1.0,47.3,26.7,-11.0,2.0,14.0,0.0,23.5,26.7,-12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,140,6796580,125697,ESL Pro League Season 23,288172,FURIA Esports,288173,paiN Gaming,2,0,...,3.0,69.8,57.9,-5.0,6.0,15.0,4.0,41.9,42.1,-9.0
141,141,6796580,-125699,ESL Pro League Season 23,288172,FURIA Esports,288173,paiN Gaming,2,0,...,9.0,67.9,55.2,-15.0,15.0,29.0,8.0,47.0,49.6,-14.0
138,138,6796582,-125701,ESL Pro League Season 23,288166,G2 Esports,288167,Aurora Gaming,0,2,...,11.0,77.5,78.6,-3.0,30.0,26.0,6.0,72.4,68.2,4.0
136,136,6796582,125701,ESL Pro League Season 23,288166,G2 Esports,288167,Aurora Gaming,0,2,...,3.0,78.0,73.9,1.0,12.0,10.0,4.0,67.6,69.6,2.0


In [15]:
df.columns

Index(['Unnamed: 0', 'match_id', 'game_id', 'tournament', 'team1_id', 'team1',
       'team2_id', 'team2', 'score1_match', 'score2_match', 'is_total',
       'bestOf', 'score1_game', 'score2_game', 'map_id', 'map_name',
       'datetime', 'team1_win', 'games_played', 'team1_player1_id',
       'team1_player2_id', 'team1_player3_id', 'team1_player4_id',
       'team1_player5_id', 'team2_player1_id', 'team2_player2_id',
       'team2_player3_id', 'team2_player4_id', 'team2_player5_id',
       'team1_player1', 'team1_player2', 'team1_player3', 'team1_player4',
       'team1_player5', 'team2_player1', 'team2_player2', 'team2_player3',
       'team2_player4', 'team2_player5', 'team1_player1_kills',
       'team1_player1_deaths', 'team1_player1_assists', 'team1_player1_adr',
       'team1_player1_kast', 'team1_player1_kddiff', 'team1_player2_kills',
       'team1_player2_deaths', 'team1_player2_assists', 'team1_player2_adr',
       'team1_player2_kast', 'team1_player2_kddiff', 'team1_player3

Let's get the average stat for the up to last 10 games this player has participated in and use that to predict if they win this game or not

In [16]:
# How many latest games do we average over
game_window = 10

#Maps player ID to dictionary of stats per game
playerDict = {}

# A fallback for if the player has no data to go off of (e.g. first time seeing them)
# Asked ChatGPT how to grab all team 1 or 2 players 1 through 5 stats
# We set our defaults to the global mean
default_stats = {
    "kills": df[df.filter(regex=r"team[12]_player[1-5]_kills").columns].sum(axis=1).sum() / (len(df) * 10),
    "deaths": df[df.filter(regex=r"team[12]_player[1-5]_deaths").columns].sum(axis=1).sum() / (len(df) * 10),
    "assists": df[df.filter(regex=r"team[12]_player[1-5]_assists").columns].sum(axis=1).sum() / (len(df) * 10),
    "adr": df[df.filter(regex=r"team[12]_player[1-5]_adr").columns].sum(axis=1).sum() / (len(df) * 10),
    "kast": df[df.filter(regex=r"team[12]_player[1-5]_kast").columns].sum(axis=1).sum() / (len(df) * 10),
    "kddiff": df[df.filter(regex=r"team[12]_player[1-5]_kddiff").columns].sum(axis=1).sum() / (len(df) * 10)
}

#Asked ChatGPT for a concise way to generate the new column names
teams = [1, 2]
players = range(1, 6)
stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]

cols = [
    f"previous_{game_window}_game_team{t}_player{p}_average_{s}"
    for t in teams
    for p in players
    for s in stats]
additional_cols = ["team1_win", 
             "team1", 
             "team2", 
             "team1_id", 
             "team2_id", 
             "game_id", 
             "match_id", 
             "tournament",
             "bestOf",
             "map_id",
             "map_name",
             "datetime"]
player_id_cols = [f"team{t}_player{p}_id"
    for t in teams
    for p in players]
cols.extend(additional_cols)
cols.extend(player_id_cols)
print(cols)

#The DataFrame we are building
entries = []
for idx, row in df.iterrows():
    engineered_entry = []
    #Iterate over both teams and their 5 players
    for k in range(1,3):
        for i in range(1,6):
            #Pull previous stats
            player_i_id = row[f"team{k}_player{i}_id"]
            player_df = playerDict.get(player_i_id)
            #Give a default stat entry if we have no recorded data for this player
            fresh_record = False
            if(player_df is None):
                player_df = [default_stats]
                fresh_record = True
            latest_window = player_df[-game_window:]
            window_df = pd.DataFrame(latest_window)
            for stat in stats:
                engineered_entry.append(window_df[stat].mean())
            #Throw out the placeholder if needed
            if(fresh_record):
                player_df.pop()
            #Add this game to the rolling window
            #Asked Gemini how to convert the row series entries to correspond to stat labels
            player_df.append({s: row[f"team{k}_player{i}_{s}"] for s in stats})
            playerDict[player_i_id] = player_df
    # Add singleton info like team name, match id, etc
    for col in additional_cols:
        engineered_entry.append(row[col])   
    # Add player ids
    for k in range(1,3):
        for i in range(1,6):
            engineered_entry.append(row[f"team{k}_player{i}_id"])
    entries.append(engineered_entry)
engineereddf = pd.DataFrame(entries, columns=cols)
engineereddf

['previous_10_game_team1_player1_average_kills', 'previous_10_game_team1_player1_average_deaths', 'previous_10_game_team1_player1_average_assists', 'previous_10_game_team1_player1_average_adr', 'previous_10_game_team1_player1_average_kast', 'previous_10_game_team1_player1_average_kddiff', 'previous_10_game_team1_player2_average_kills', 'previous_10_game_team1_player2_average_deaths', 'previous_10_game_team1_player2_average_assists', 'previous_10_game_team1_player2_average_adr', 'previous_10_game_team1_player2_average_kast', 'previous_10_game_team1_player2_average_kddiff', 'previous_10_game_team1_player3_average_kills', 'previous_10_game_team1_player3_average_deaths', 'previous_10_game_team1_player3_average_assists', 'previous_10_game_team1_player3_average_adr', 'previous_10_game_team1_player3_average_kast', 'previous_10_game_team1_player3_average_kddiff', 'previous_10_game_team1_player4_average_kills', 'previous_10_game_team1_player4_average_deaths', 'previous_10_game_team1_player4_ave

,previous_10_game_team1_player1_average_kills,previous_10_game_team1_player1_average_deaths,previous_10_game_team1_player1_average_assists,previous_10_game_team1_player1_average_adr,previous_10_game_team1_player1_average_kast,previous_10_game_team1_player1_average_kddiff,previous_10_game_team1_player2_average_kills,previous_10_game_team1_player2_average_deaths,previous_10_game_team1_player2_average_assists,previous_10_game_team1_player2_average_adr,...,team1_player1_id,team1_player2_id,team1_player3_id,team1_player4_id,team1_player5_id,team2_player1_id,team2_player2_id,team2_player3_id,team2_player4_id,team2_player5_id
0,16.919962,17.041386,5.919138,72.387668,71.542701,-0.121423,16.919962,17.041386,5.919138,72.387668,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
1,13.000000,15.000000,6.000000,61.900000,82.600000,-2.000000,16.000000,13.000000,5.000000,81.600000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
2,12.500000,16.000000,5.000000,60.500000,72.550000,-3.500000,14.500000,12.000000,3.500000,67.100000,...,553.0,39.0,612.0,163.0,457.0,739.0,1443.0,65.0,2049.0,209.0
3,16.919962,17.041386,5.919138,72.387668,71.542701,-0.121423,16.919962,17.041386,5.919138,72.387668,...,3202.0,1420.0,3168.0,607.0,3032.0,3836.0,1886.0,2588.0,1778.0,4567.0
4,30.000000,16.000000,7.000000,90.800000,93.400000,14.000000,27.000000,20.000000,10.000000,74.300000,...,3202.0,1420.0,3168.0,607.0,3032.0,3836.0,1886.0,2588.0,1778.0,4567.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6908,14.100000,14.900000,7.000000,66.110000,77.490000,-0.800000,16.800000,17.000000,8.200000,76.090000,...,625.0,628.0,1.0,26759.0,564.0,690.0,1871.0,5620.0,1518.0,887.0
6909,14.500000,14.000000,6.900000,66.420000,80.250000,0.500000,16.400000,16.300000,8.800000,76.890000,...,625.0,628.0,1.0,26759.0,564.0,690.0,1871.0,5620.0,1518.0,887.0
6910,14.600000,16.400000,6.900000,66.290000,76.660000,-1.800000,16.400000,19.000000,7.400000,80.950000,...,416.0,881.0,3459.0,4048.0,1661.0,157.0,156.0,197.0,4903.0,4896.0
6911,14.900000,17.600000,7.400000,64.970000,75.240000,-2.700000,17.300000,20.100000,7.800000,80.830000,...,416.0,881.0,3459.0,4048.0,1661.0,157.0,156.0,197.0,4903.0,4896.0


In [17]:
engineereddf.columns

Index(['previous_10_game_team1_player1_average_kills',
       'previous_10_game_team1_player1_average_deaths',
       'previous_10_game_team1_player1_average_assists',
       'previous_10_game_team1_player1_average_adr',
       'previous_10_game_team1_player1_average_kast',
       'previous_10_game_team1_player1_average_kddiff',
       'previous_10_game_team1_player2_average_kills',
       'previous_10_game_team1_player2_average_deaths',
       'previous_10_game_team1_player2_average_assists',
       'previous_10_game_team1_player2_average_adr',
       'previous_10_game_team1_player2_average_kast',
       'previous_10_game_team1_player2_average_kddiff',
       'previous_10_game_team1_player3_average_kills',
       'previous_10_game_team1_player3_average_deaths',
       'previous_10_game_team1_player3_average_assists',
       'previous_10_game_team1_player3_average_adr',
       'previous_10_game_team1_player3_average_kast',
       'previous_10_game_team1_player3_average_kddiff',
       

# We can now save our engineered dataset!

In [18]:
engineereddf.to_csv("../../data/rolling_feature_engineered_tier_1_games.csv", index=False)
print("Saved to data/rolling_feature_engineered_tier_1_games.csv")

Saved to data/rolling_feature_engineered_tier_1_games.csv
